# Convols: Build the Multiresolution Field

This notebook shows how PyHermes converts particle data into `ConvolsData`, the multiresolution field representation used by later tasks such as counting, 2PCF, and 3PCF. The same machinery also lets you change particle weights to build different physical fields from the same catalog.

## What this notebook emphasizes

- how supported particle readers behave on real inputs
- how to run `Convols` from progressively lower-level interfaces
- how unit field values build a catalogue-normalized tracer field, while mass values build a mass-valued field
- how to convert a velocity catalog from real space to redshift space before building the field
- how to build a matching random field for later `DR` / `RR` calculations
- how window convolution is applied once the field has been built

## Suggested reading order

A typical walkthrough is:

1. `convols.ipynb` to build the field
2. `window.ipynb` to learn field/window algebra and smoothing filters
3. `counting.ipynb` to sample the field at random points
4. `corr2pcf.ipynb` to measure two-point statistics
5. `corr3pcf.ipynb` to measure three-point statistics
6. `weighted_fields.ipynb` as an extra application beyond the traditional multipoint-statistics workflow, building velocity and momentum-density fields by changing weights


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pyhermes.base.convols import Convols
from pyhermes.param.parambase import read_param
from pyhermes.utils.sampling import random_box_positions, regular_grid_positions
from pyhermes.utils.func_util import validate_convols_compatibility
from pyhermes.utils.redshift_space import hubble_at_redshift, redshift_space_positions
from pyhermes.io import ConvolsData, read_particle_data

from pathlib import Path
import os
os.chdir(Path.cwd().resolve().parent)
print(f"Working directory: {Path.cwd()}")

## Optional: download the example halo catalog

If `./data/quijote_halos/` is not available locally, run the next cell to download the archive from the PyHermes documentation site into `examples/data/` and extract it in place.


In [ ]:
!mkdir -p ./data
!curl -L \
  -A 'Mozilla/5.0' \
  -e 'https://pyhermes.astroslacker.com/' \
  -o ./data/quijote_halos.tar.gz \
  'https://pyhermes.astroslacker.com/_downloads/87691d6e7eb8dd0b954576b2bc71fb51/quijote_halos.tar.gz'

!tar -xzf ./data/quijote_halos.tar.gz -C ./data --exclude='._*' --exclude='__MACOSX'


## 1. Read particle data directly

The next few cells are reader checks. They show that PyHermes can ingest the same dataset through different entry points before any multiresolution field is constructed.


In [ ]:
reader_params = {
    "snapnum": 4,
    "redshift": 0.0,
    "fields": {
        "vel": "vel",
        "vx": "vel_x",
        "vy": "vel_y",
        "vz": "vel_z",
        "mass": "mass",
        "npart": "npart",
    }
}
fof_data = read_particle_data("./data/quijote_halos/8000", data_format="fof", **reader_params)
fof_data


### Optional: Repack the FoF halo catalog into the documented raw binary table

The Quijote release ships the original `group_tab` catalog, not the compact `.bin` table used in some PyHermes examples. The next cell reads `./data/quijote_halos/quijote_halo_bin_schema.yaml`, follows its documented column order, and writes a dense float32 table that can be consumed by the generic `bin` reader.


In [ ]:


schema_path = Path("./data/quijote_halos/quijote_halo_bin_schema.yaml")
bin_path = Path("./data/quijote_halos/8000/groups_004/group_tab_004.bin")

schema = yaml.safe_load(schema_path.read_text())
columns = schema["columns"]

column_arrays = {
    "x": fof_data["pos"][:, 0],
    "y": fof_data["pos"][:, 1],
    "z": fof_data["pos"][:, 2],
    "vx": fof_data["vx"],
    "vy": fof_data["vy"],
    "vz": fof_data["vz"],
    "mass": fof_data["mass"],
    "npart": fof_data["npart"],
}

missing = [name for name in columns if name not in column_arrays]
if missing:
    raise KeyError(f"Cannot build the binary table because these schema columns are missing: {missing}")

table = np.column_stack([
    np.asarray(column_arrays[name], dtype=np.float32)
    for name in columns
]).astype(np.float32, copy=False)

if table.shape[1] != schema["ncols"]:
    raise ValueError(
        f"Schema expects {schema['ncols']} columns, but the packed table has shape {table.shape}."
    )

bin_path.parent.mkdir(parents=True, exist_ok=True)
table.tofile(bin_path)
print(f"Wrote {bin_path} with shape {table.shape} and dtype {table.dtype}.")


In [ ]:
reader_params = {
    "dtype": schema["dtype"],
    "ncols": schema["ncols"],
    "pos_cols": [0, 1, 2],
    "fields": {
        "vel": [3, 4, 5],
        "vx": 3,
        "vy": 4,
        "vz": 5,
        "mass": 6,
        "npart": 7,
    }
}
bin_data = read_particle_data(str(bin_path), **reader_params)
{
    "size_match": bin_data["size"] == fof_data["size"],
    "pos_match": np.allclose(bin_data["pos"], fof_data["pos"]),
    "vel_match": np.allclose(bin_data["vel"], fof_data["vel"]),
    "mass_match": np.allclose(bin_data["mass"], fof_data["mass"]),
    "npart_match": np.array_equal(bin_data["npart"], fof_data["npart"].astype(np.float32)),
}


## 2. Build `ConvolsData`

`Convols` is the upstream stage of the whole workflow. It takes particle positions, projects them into the PyHermes multiresolution basis, and returns a field object that can later be sampled, convolved, or correlated.

### Core idea

- `box_size` sets the physical domain size.
- `J` controls the multiresolution level.
- `wavelet_mode`, `wavelet_level`, and `phi_resolution` control the basis construction.
- the output `ConvolsData` object stores both the field values and the metadata required by later window operations.


### Minimal YAML Shapes

`Convols` builds the saved multiresolution field used by the later notebooks. A minimal config specifies the input reader, the box/grid parameters, the wavelet basis, and the output path:

```yaml
Convols:
   fin:
      path: "./data/quijote_halos/8000"
      format: "fof"
      reader_params:
         snapnum: 4
   box_size: 1000
   J: 8
   wavelet_mode: "db2"
   wavelet_level: 10
   phi_resolution: 1024
   threads: 2
   fout_path: "./output_new/quijote8000_snap004_sfc.pkl"
```

With unit field values, the resulting field is a catalogue-normalized tracer-density field whose integral is one. `catalog_weight_key` represents relative observational or selection weights and is automatically normalized by its sum, while `field_value_key` selects the physical quantity carried by each halo. Setting `field_value_key: "mass"` builds a mass-valued field with catalogue-weighted mean mass amplitude without changing the catalogue measure. The later `weighted_fields.ipynb` notebook uses the same distinction for velocity and momentum-density fields.


### Command-line entry point

This is the most direct way to run `Convols` in production. A YAML file defines the task, and the standard driver script handles the run.


In [ ]:
! mpirun -np 4 python ./scripts/run_convols.py ./configs/param_convols.yaml

### Config-driven Python API

This version still treats the YAML config as the main source of truth, but launches the task from Python so that you can inspect objects in the notebook.


In [ ]:
convols_params = read_param(config_path="./configs/param_convols.yaml")
task = Convols(param_task=convols_params)
task.threads = 8
task.save_particle_data = False
convols_data = task.run(save_result=False)

### Task object overrides

Here the task object is created first and then adjusted in Python. This is useful when the config is close to what you want, but a few runtime settings need to change.

The next two cells intentionally create two independent `Convols()` objects: one for a higher-resolution `J=9` field, and one for a mass-valued field. The first run represents the unit-integral halo catalogue density; the second carries mass through `field_value` on the same normalized catalogue measure. Treat each saved field as its own task run. If particle inputs, values, grid parameters, or reader settings change, start from a fresh task object instead of reusing a previously prepared one.


In [ ]:
task = Convols()
task.fin = {
    "path": "./data/quijote_halos/8000",
    "format": "fof",
    "reader_params": {
        "snapnum": 4
    }
}
task.J = 9
task.threads = 8
task.fout_path = "./output_new/quijote8000_snap004_sfc_J9.pkl"
task.run(overwrite=True)

In [ ]:
task = Convols()
task.fin = {
    "path": "./data/quijote_halos/8000",
    "format": "fof",
    "reader_params": {
        "snapnum": 4,
        "fields": {
            "mass": "mass",
        }
    },
    "field_value_key": "mass"
}
task.threads = 8
task.fout_path = "./output_new/quijote8000_snap004_sfc_massweight.pkl"
task.run(overwrite=True)

### Manual particle input and redshift-space preparation

At this level the particle positions are prepared explicitly and injected into the task object. This is the most flexible route when your upstream data preparation is custom.

It is also the most natural place to insert a real-space to redshift-space mapping: load positions and velocities, shift the particles along a chosen line of sight, and then run `Convols` on the transformed catalog.

As above, each output is built with a fresh `Convols()` instance. This keeps the prepared particle arrays, catalogue weights, physical field values, and metadata tied to exactly one saved field.


In [ ]:
reader_params = {
    "snapnum": 4,
    "redshift": 0.0,
    "fields": {
        "vel": "vel",
        "mass": "mass",
    }
}
data = read_particle_data("./data/quijote_halos/8000", data_format="fof", **reader_params)
hubble_parameter = hubble_at_redshift(redshift=0)

#### Build redshift-space fields for different lines of sight

The next cells use `redshift_space_positions(...)` to transform the same halo sample into redshift space and then build `ConvolsData` from the shifted positions.

- first with `los="z"` and unit field values for a box-axis line of sight
- then with the same z-axis redshift-space positions but mass weights
- finally with `los=[1, 1, 1]` for a diagonal line of sight

These outputs are useful downstream because anisotropic statistics such as `(s, mu)` 2PCF are sensitive to both the presence of redshift-space distortions and the chosen viewing direction. The unit-value output is a redshift-space catalogue-normalized tracer field, while the mass-valued output carries mass marks on the same catalogue measure; comparing them checks how physical marks change later measurements.


In [ ]:
pos = redshift_space_positions(data['pos'], data['vel'], box_size=1000,
                               hubble=hubble_parameter, redshift=0, los="z")
task = Convols()
task.particle_pos = pos
task.threads = 8
task.fout_path = "./output_new/quijote8000_snap004_rsd_sfc.pkl"
task.run(overwrite=True)

In [ ]:
task = Convols()
task.particle_pos = pos
task.field_value = data["mass"]
task.threads = 8
task.fout_path = "./output_new/quijote8000_snap004_rsd_sfc_massweight.pkl"
task.run(overwrite=True)

In [ ]:
pos = redshift_space_positions(data['pos'], data['vel'], box_size=1000,
                               hubble=hubble_parameter, redshift=0, los=[1, 1, 1])
task = Convols()
task.particle_pos = pos
task.threads = 8
task.fout_path = "./output_new/quijote8000_snap004_rsd_diag_sfc.pkl"
task.run(overwrite=True)

## 3. Build a matching random field

Many downstream estimators compare the data field to a random field with the same grid and metadata. The next cell constructs such a random catalog and stores it as a second `ConvolsData` object.


In [ ]:
random_pos = random_box_positions(count=10_000_000, box_size=1000, seed=42)
task = Convols()
task.particle_pos = random_pos
task.fout_path = "./output_new/random_sfc.pkl"
task.save_particle_data = True
task.particle_data_path = "./data/random_1e7.npz"
task.threads = 8
task.run(overwrite=True)

## 4. Reload the data and random fields

Once both fields exist on disk, PyHermes can reload them and verify that their required metadata are compatible. This compatibility check matters because subtraction, pair products, and later correlation estimators assume that both fields live on the same grid.


In [ ]:
D = ConvolsData(data_path='./output_new/quijote8000_snap004_sfc.pkl', threads=8)
R = ConvolsData(data_path='./output_new/random_sfc.pkl', threads=8)

In [ ]:
shared_required = validate_convols_compatibility([D, R], ConvolsData._REQUIRED_ARGV)
print("Compatibility check passed. Shared required parameters:")
print(", ".join([f"{k}={v}" for k, v in shared_required.items()]))

## 5. Visualize the projected field on a two-dimensional slice

The final diagnostic compares the catalogue-normalized multiresolution field values at three resolutions with the underlying halo positions. We use `regular_grid_positions` to identify a central grid slab, project the J=7, J=8, and J=9 `epsilon` fields through that slab, and show the halo scatter over the same slice.


In [ ]:
convols_params = read_param(config_path="./configs/param_convols.yaml")
task = Convols(param_task=convols_params)
task.J = 7
task.threads = 8
D7 = task.run(save_result=False)

D8 = ConvolsData(data_path='./output_new/quijote8000_snap004_sfc.pkl', threads=8)
D9 = ConvolsData(data_path='./output_new/quijote8000_snap004_sfc_J9.pkl', threads=8)
halo_pos = read_particle_data('./data/quijote_halos/8000', data_format='fof', snapnum=4)['pos']

In [ ]:
box_size = 1000.0
x_range = (300.0, 700.0)
y_range = (300.0, 700.0)
slice_z = 500.0
slice_thickness = 50.0


def periodic_slab_mask(values, center, thickness, box_size):
    periodic_delta = (values - center + 0.5 * box_size) % box_size - 0.5 * box_size
    return np.abs(periodic_delta) <= 0.5 * thickness


def slice_and_cut_field(field):
    axis, _ = regular_grid_positions(field.L, field.box_size, ndim=1)
    axis = axis[:, 0]

    x_mask = (axis >= x_range[0]) & (axis <= x_range[1])
    y_mask = (axis >= y_range[0]) & (axis <= y_range[1])
    z_mask = periodic_slab_mask(axis, slice_z, slice_thickness, box_size)

    eps_cut = field.epsilon[np.ix_(x_mask, y_mask, z_mask)].sum(axis=2)

    return eps_cut

def slice_and_cut_pos(halo_pos):
    halo_mask = (
        periodic_slab_mask(halo_pos[:, 2], slice_z, slice_thickness, box_size)
        & (halo_pos[:, 0] >= x_range[0]) & (halo_pos[:, 0] <= x_range[1])
        & (halo_pos[:, 1] >= y_range[0]) & (halo_pos[:, 1] <= y_range[1])
    )
    return halo_pos[halo_mask]


eps7 = slice_and_cut_field(D7)
eps8 = slice_and_cut_field(D8)
eps9 = slice_and_cut_field(D9)
halo_cut = slice_and_cut_pos(halo_pos)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True)

extent = (*x_range, *y_range)

panels = [
    (axes[0, 0], eps7, 'A: J=7'),
    (axes[0, 1], eps8, 'B: J=8'),
    (axes[1, 0], eps9, 'C: J=9'),
]

for ax, image, label in panels:
    positive = image[image > 0]
    vmax = np.percentile(positive, 99.7) if positive.size else np.percentile(image, 99.7)

    ax.imshow(image.T, origin='lower', extent=extent, cmap='Greys', vmin=0, vmax=vmax, interpolation='nearest')
    ax.text(0.95, 0.95, label, transform=ax.transAxes, ha='right', va='top', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='green', linewidth=1.2))

axes[1, 1].scatter(halo_cut[:, 0], halo_cut[:, 1], s=3.0, c='black', alpha=0.75, linewidths=0)
axes[1, 1].text(0.95, 0.95, 'D: halo scatter', transform=axes[1, 1].transAxes, ha='right', va='top', fontsize=12,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='red', linewidth=1.2))

for ax in axes.ravel():
    ax.set_xlim(*x_range)
    ax.set_ylim(*y_range)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=10)
    ax.label_outer()

fig.supxlabel(r'$x\ [h^{-1}\mathrm{Mpc}]$', fontsize=14)
fig.supylabel(r'$y\ [h^{-1}\mathrm{Mpc}]$', fontsize=14)
fig.tight_layout(w_pad=1.0, h_pad=1.0)

figs_dir = Path('figs_new')
figs_dir.mkdir(exist_ok=True)
plt.savefig(figs_dir / 'convols_epsilon_slice_j7_j8_j9_scatter.png', dpi=200, bbox_inches='tight')

plt.show()